In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
import random
import os
import pandas as pd

from matplotlib import pyplot as plt

from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx

import sys
sys.path.append("..")

from tools.small_model import FC_MD

os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:

import statsmodels.api as sm
from scipy.integrate import simps

d = [-100, -99, 0.1, 0.2, 0.3]

ecdf = sm.distributions.ECDF(d)
x = np.linspace(-50, 1, num=1000)
y = ecdf(x)

print(y)

In [ ]:
x = torch.tensor([[[1,2,3],[3,2,4]]])
print(x)
x = x.view(-1, 6) 
print(x)

torch.cat((x,x), axis=1).shape

In [ ]:
seed = 59

# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
data_train = MNIST('../data/mnist',
                  train=True,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))

data_test = MNIST('../data/mnist',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))


In [ ]:
file_path = "edge_v/"
model_path = "models/"

In [ ]:
def get_new_data(l1):
    # selected classes
    train_i1 = torch.tensor([i for i, (_, label) in enumerate(data_train) if label in l1])
    test_i1 = torch.tensor([i for i, (_, label) in enumerate(data_test) if label in l1])
    
    train_index = torch.randperm(len(train_i1))
    valid_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[0:5000]])
    train_dataset = torch.utils.data.Subset(data_train, train_i1[train_index[5000:,]])
    test_dataset = torch.utils.data.Subset(data_test, test_i1)
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=1000, num_workers=2)
    valid_loader = DataLoader(valid_dataset, batch_size=1000, num_workers=2)
    
    return train_loader, test_loader, valid_loader, valid_dataset

In [ ]:
def sep_label(dataset, ls):
    sep_dataloader = dict()
    for l in ls:
        index = torch.tensor([i for i, (_, label) in enumerate(dataset) if label == l])
        subset = torch.utils.data.Subset(dataset, index)
        loader = DataLoader(subset, batch_size=5000, num_workers=2)

        sep_dataloader[l] = loader
        
    return sep_dataloader

In [ ]:
selected_classes = [0,1,2,3,4,5,6,7,8,9]
train_loader, test_loader, valid_loader, valid_dataset = get_new_data(selected_classes)

sep_dataloader = sep_label(valid_dataset, selected_classes)


In [ ]:
'''
[784, 20, 15, 10]
[784, 15, 25, 20, 15, 10]
[784, 20, 30, 30, 20, 15, 10]
[784, 20, 30, 30, 35, 20, 15, 10]
[784, 30, 30, 40, 50, 30, 25, 20, 10]
'''

dims = [784, 20, 15, 10]
layer_num = 2  # [2, 4, 5, 6, 7]

model_name = "best_ori_10l_" + str(layer_num) + ".pth"

net_H = FC_MD(dims, layer_num)

net_H.load_state_dict(torch.load(model_path + model_name))
net_H = net_H.to(device)

In [ ]:
neural_list = []
nodes_num = 0
edges_num = 0
i = 0
for p in net_H.parameters():
    if i == 0:
        nodes_num += p.shape[1]
    if i%2 == 0:
        nodes_num += p.shape[0]
        edges_num += (p.shape[0] * p.shape[1])
        neural_list.append(p.shape[0])
    i += 1
    
print(f'Total nodes are {nodes_num}, total edges are {edges_num}.')

In [ ]:
import csv
import pandas as pd

# Extract activate values using all the test set
# labels_n = [0,1,2,3,4,5,6,7,8,9]
# sample_l = np.random.choice(labels_n, 1)

# model_n = "less_class_" + str(l) + ".pth"
net_H.load_state_dict(torch.load(model_path + model_name))
net_H = net_H.to(device)
net_H.eval()

loader = valid_loader
for i, (d, label) in enumerate(loader):    
    d = d.to(device)
    output = net_H.edge_w_batch(d)
    
    output = output.cpu().detach().numpy() 
    if (i == 0):
        data = output
    else:
        data = np.vstack((data,output))
                
data_f = data.tolist()
edge_weights = pd.DataFrame(data=data_f)
edge_weights.to_csv(file_path + "full_edge_v.csv")

In [ ]:
avg_edge_w = []

for l in [0]:
    file_n = "full_edge_v_" + str(layer_num) + ".csv"
    # file_n = "edge_v" + str(l) + ".csv"
    # raw value: edge weights
    edge_weights = pd.read_csv(file_path + file_n, index_col=0)
    # cols = edge_weights.columns # number of neurals
    e_weights = edge_weights.values
    w_avg = np.mean(e_weights, axis=0)
    avg_edge_w.append((l, w_avg))

In [ ]:
# Function to perform min-max scaling on weights
def min_max_scale(weights, min_val, max_val):
    scaled_weights = (weights - min_val) / (max_val - min_val)
    return scaled_weights

In [ ]:
scaled_weights = []
abs_weights = []
for (l, w_avg) in avg_edge_w:
    # w_avg = np.abs(w_avg) 
    # get min max
    min_w = min(w_avg)
    max_w = max(w_avg)
    print(f"Min: {min_w:.2f}, Max: {max_w:.2f}")

    scaled_weights.append((l, min_max_scale(w_avg, min_w, max_w)))
    abs_weights.append((l, w_avg))


In [ ]:
# plt.figure()
# for (l, w_avg) in abs_weights:
#     n, bins, patches = plt.hist(w_avg, bins=[0, 0.01, 0.1, 0.5, max(w_avg)], alpha=0.5, label=str(l))
    
#     print(n)
#     print(bins)

#     plt.xlabel('Edge value')
#     plt.title("Histogram of Positive Edge values")
#     plt.legend()
#     plt.show()

In [ ]:
adjacent_m_l = []

for (l, sca_w) in abs_weights:
    # build adjacent matrix
    adjacent_m = np.zeros((nodes_num, nodes_num), dtype=np.float32)

    layer_num = len(dims)
    cur_s_col = dims[0]
    cur_e_col = dims[0] + dims[1]

    cur_layer = 1
    start_col = 0
    end_col = dims[1]

    for i in range(nodes_num - dims[layer_num-1]):
        # print(f'i : {i}, start col : {cur_s_col}, end_col : {cur_e_col}, from {start_col} to {end_col}')
        adjacent_m[i, cur_s_col : cur_e_col] = sca_w[start_col : end_col]
        
        if (cur_layer < layer_num-1 and i == cur_s_col - 1):
            cur_layer += 1
            start_col = end_col
            end_col = end_col + dims[cur_layer]
            cur_s_col = cur_e_col
            cur_e_col = cur_e_col + dims[cur_layer]
        else:
            start_col = end_col
            end_col = end_col + dims[cur_layer]
            
    adjacent_m_l.append((l, adjacent_m))
        


In [ ]:
def show_results(G, label, curvature="ricciCurvature"):
    
    neg_e = 0
    edge_set = set()
    most_pos = 0
    nodes = (0,0)
    curs = []

    # Print the first five results
    for n1,n2 in list(G.edges()):
        if (G[n1][n2][curvature] > 0.95):
            curs.append(G[n1][n2][curvature])
            neg_e += 1
            edge_set.add((n1, n2))
            if (G[n1][n2][curvature] > most_pos):
                most_pos = G[n1][n2][curvature]
                nodes = (n1, n2)
            # print("Ricci curvature of edge (%s,%s) is %f" % (n1 ,n2, G[n1][n2][curvature]))
    print(f'Lable: {label}: Total number of edges have positive curvature is {neg_e}')
    print(f'The most positive curvature is {most_pos}, between nodes {nodes[0]} and {nodes[1]}, {G[nodes[0]][nodes[1]]["weight"]}.')
    # Plot the histogram of Ricci curvatures
    # plt.subplot(2, 1, 1)
    # ricci_curvtures = nx.get_edge_attributes(G, curvature).values()
    # plt.hist(curs, bins=80)
    # plt.xlabel(f'Ricci curvature: label {label}')
    # plt.title("Histogram of Ricci Curvatures")

    # Plot the histogram of edge weights
    # plt.subplot(2, 1, 2)
    # weights = nx.get_edge_attributes(G, "weight").values()
    # plt.hist(weights,bins=20)
    # plt.xlabel(f'Edge weight for label {label}')
    # plt.title("Histogram of Edge weights")

    # plt.tight_layout()
    # plt.show()
    
    return edge_set, nodes, curs

In [ ]:
G_l = []
for (l, adjacent_m) in adjacent_m_l:
    # Create network object
    G = nx.from_numpy_array(adjacent_m, create_using=nx.DiGraph)

    orf = OllivierRicci(G, alpha=0.5, verbose="ERROR")
    # orf.compute_ricci_flow(iterations=100)
    orf.compute_ricci_curvature()

    G1 = orf.G.copy()
    print(G1)

    G_l.append((l, G, G1))

In [ ]:
e_l = []
curs_l = []
for (l, G, G1) in G_l:
    # edge = sorted(G1.edges(data=True), key=lambda edge: edge[2].get("ricciCurvature", 0), reverse = True)
    edge_set, most_nodes, curs = show_results(G1, l, "ricciCurvature")
    e_l.append((edge_set, most_nodes, l))
    curs_l.append((l, curs))


In [ ]:
plt.figure()
for (l, curs) in curs_l:
    if(l == 0 or l == 5):
        n, bins, patches = plt.hist(curs, bins=20, alpha=0.5, label=str(l))
        print(n)
    
        plt.xlabel('Ricci curvature')
        plt.title("Histogram of Positive Ricci Curvatures")
        plt.legend()
        plt.show()

In [ ]:
base_edge_s = e_l[0][0]

common_e = base_edge_s & e_l[1][0]
diff_e_l = []

for i in range(2, len(e_l), 1):
    common_e = common_e & e_l[i][0]

for (edge_set, most_nodes, l) in e_l:
    cur_e_s = edge_set
    cur_e_s = cur_e_s - common_e
    diff_e_l.append((l, cur_e_s))

In [ ]:
for (l,sl) in diff_e_l:
    print(len(sl))
    
print(len(common_e))

In [ ]:
# from edge_remove import Edge_Remove

# new_path = model_path

# # remove common edges: # 1
# net_H.load_state_dict(torch.load(model_name))
# edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)

# label_num = len(neg_e_l)

# edge_r.e_remove(common_e, "common")

# # remove all 10 classes edges: #1
# net_H.load_state_dict(torch.load(model_name))
# edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)

# for (edge_set, most_nodes, l) in neg_e_l:
#     edge_r.e_remove(edge_set, "all")

# # remove each class edges: # 10
# for (edge_set, most_nodes, l) in neg_e_l:
#     net_H.load_state_dict(torch.load(model_name))
#     edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)
#     edge_r.e_remove(edge_set, "class" + str(l))

# # remove (each class edges - common edges): # 10
# for (l, edge_set) in diff_e_l:
#     net_H.load_state_dict(torch.load(model_name))
#     edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", new_path)
#     edge_r.e_remove(edge_set, "diff" + str(l))

1. XOR curvature

2. 1/edge_weight

3. pick top 1000 hightest edge weight edges

In [ ]:
# remove top 1000 highest edge weights
# 10 classes
# path = "top1000/"
# edges = []

# for (l, G, G1) in G_l:
#     edge = sorted(G1.edges(data=True), key=lambda edge: edge[2].get('weight', 0), reverse = True)[0:1000]
#     edge = list(np.array(edge)[:, 0:2])
#     my_set = {(n1,n2) for (n1,n2) in edge}

#     edges.append((l, my_set))
    

In [ ]:
# compute pair
# 10 labels
# 45 pairs

# from edge_remove import Edge_Remove

# for (edge_set1, most_nodes1, i) in e_l:
#     for (edge_set2, most_nodes2, j) in e_l:
#         if (j != i):
#             common_e = edge_set1 & edge_set2
#             diff_ = edge_set2 - common_e
            
#             net_H.load_state_dict(torch.load(model_path + model_name))
#             edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", model_path)
#             edge_r.e_remove(diff_, "pair_" + str(j) + "_" + str(i) + str(j))
            
#             net_H.load_state_dict(torch.load(model_path + model_name))
#             edge_r = Edge_Remove(net_H, dims, 28, G1, "ricciCurvature", model_path)
#             edge_r.e_remove(common_e, "common_" + str(i) + str(j))


1. plot graph

2. pgd: images chaged send to , recalculate edge weight